In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [4]:
doc_idx

{'74eb249bbf': {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 '977bf7786c': {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 '489dd1c9d9': {'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours

In [5]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [6]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [7]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join if you just discovered the course.\n\nIf you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

In [8]:
rec

{'question': 'I found this course late — can I still enroll and participate, or is it too late now?',
 'document': '74eb249bbf'}

In [12]:
documents = load_faq_data()
documents

[{'id': 'b043d949b7',
  'course': 'stock-markets-analytics-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Do I need to pay for any platform or data for this course?',
  'answer': 'No. All the data sources used in the course (yfinance, pandas_datareader, web scraping) are free, and you do not need a paid GCP account or any other paid service. You are welcome to use paid data if you want, but it is not required.'},
 {'id': '6f669a2215',
  'course': 'stock-markets-analytics-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Do I need a finance background to take this course?',
  'answer': 'It helps, but you can start without one - though "no finance experience needed" turns out to be a bit optimistic, and you will need to read up on terms as you go. Good free resources: Investopedia for definitions, ChatGPT for explanations, Khan Academy\'s economics-finance course, and Zerodha Varsity. Take it one concept at a time rather than trying to

In [13]:
documents[0]

{'id': 'b043d949b7',
 'course': 'stock-markets-analytics-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Do I need to pay for any platform or data for this course?',
 'answer': 'No. All the data sources used in the course (yfinance, pandas_datareader, web scraping) are free, and you do not need a paid GCP account or any other paid service. You are welcome to use paid data if you want, but it is not required.'}

In [21]:

for doc in documents:
    if doc["id"] == "74eb249bbf":
        print(doc)

{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}


In [16]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes — you can still join if you just discovered the course.\n\nIf you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [17]:
assistant.total_cost()

0.0011279999999999999

In [18]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [20]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I found this course late — can I still enroll and participate, or is it too late now?',
 'answer_llm': 'Yes — you can still join if you just discovered the course.\n\nIf you want to receive a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [22]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [23]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I found this course late — can I still enroll and participate, or is it too late now?',
 'answer_llm': 'Yes, you can still join and participate. If you want to receive a certificate, you need to submit your project while the course is still accepting submissions.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [24]:
assistant.reset_usage()

In [25]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [26]:
ground_truth

[{'question': 'I found this course late — can I still enroll and participate, or is it too late now?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course after it already started, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do latecomers get to take part in the course, or are registrations closed completely?',
  'document': '74eb249bbf'},
 {'question': 'What’s the cutoff for getting a certificate if I’m just discovering the course now?',
  'document': '74eb249bbf'},
 {'question': 'Can I still join the course after it has begun, and what do I need to do to get certified?',
  'document': '74eb249bbf'},
 {'question': 'Do I actually need a confirmation email to start the LLM Zoomcamp, or am I already in?',
  'document': '977bf7786c'},
 {'question': 'If I signed up for the course, should I still wait for an acceptance email before watching the lessons or doing homework?',
  'document': '977bf7786c'},
 {'question': 'Is there any list tha

In [27]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/765 [00:00<?, ?it/s]

In [28]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [29]:
answers

[{'question': 'I found this course late — can I still enroll and participate, or is it too late now?',
  'answer_llm': 'Yes — you can still join and participate.\n\nFrom the course FAQ: if you just discovered the course, you can still join, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join the course after it already started, am I still eligible for a certificate?',
  'answer_llm': 'Yes, you can still join after the course has started, but to be eligible for a certificate you must submit your project while a live cohort is still accepting submissions.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {

In [30]:
assistant.total_cost()

0.8720115000000003

In [31]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)